!pip install opencv-python numpy matplotlib

| Given Problem / Image mein kya hai?                               | Suitable Method(s)             | Easy Reason / Why?                                                                                       |
| ----------------------------------------------------------------- | ------------------------------ | -------------------------------------------------------------------------------------------------------- |
| **Simple image, clear dark & bright regions**                     | **Global Thresholding**        | Ek fixed threshold se object aur background easily separate ho jate hain.                                |
| **Uneven lighting / shadows**                                     | **Adaptive Thresholding** ⭐    | Har local area ka different threshold use karta hai, isliye shadows/brightness changes handle karta hai. |
| **Clear foreground & background intensity difference**            | **Otsu Thresholding** ⭐        | Automatically best global threshold find karta hai.                                                      |
| **Colored object / specific color identify karna ho**             | **Color-based Thresholding** ⭐ | Specific color range ko isolate karta hai. HSV mein usually easier hota hai.                             |
| **Touching/overlapping objects ko separate karna ho**             | **Watershed** ⭐                | Touching objects ko alag regions mein divide karne ke liye useful hai.                                   |
| **Object aur background ka color/intensity consistent ho**        | **K-Means**                    | Similar-color pixels ko different clusters mein divide karta hai.                                        |
| **Image mein clear edges/boundaries hon**                         | **Canny Edge Detection** ⭐     | Object boundaries/edges detect karta hai.                                                                |
| **Object ke andar intensity/color relatively similar ho**         | **Region Growing** ⭐           | Seed point se similar pixels ko grow karke region banata hai.                                            |
| **Natural scene with many colors/regions**                        | **K-Means / Color-based**      | Different colors ko clusters/regions mein divide kar sakte hain.                                         |
| **Grayscale object + background with clear intensity difference** | **Global / Otsu**              | Intensity ke basis par object aur background separate ho jate hain.                                      |
| **Grayscale image + uneven illumination**                         | **Adaptive Thresholding**      | Local threshold ki wajah se brightness variation handle hoti hai.                                        |
| **Multiple touching objects + clear foreground**                  | **Watershed**                  | Foreground markers bana kar touching objects separate karta hai.                                         |
| **Specific region similar to seed point**                         | **Region Growing**             | Seed se start karke similar-intensity pixels add karta hai.                                              |
| **Need to reduce many colors into few representative colors**     | **K-Means**                    | Image ke colors ko K clusters mein simplify karta hai.                                                   |



Lab 05

# ============================================================
# CANNY EDGE DETECTION
# Identifying:
# 1. Strong Edges
# 2. Weak Edges
# 3. Missing Edges
# 4. Unwanted Edges
# ============================================================

import cv2
import numpy as np
from matplotlib import pyplot as plt


# ============================================================
# STEP 1: READ THE IMAGE
# ============================================================

# cv2.imread() reads an image from the given file path.
# The image is loaded in BGR format by OpenCV.
image = cv2.imread('dog.jpeg')


# ============================================================
# STEP 2: CONVERT IMAGE TO GRAYSCALE
# ============================================================

# Canny edge detection normally works on a grayscale image.
#
# cv2.cvtColor() is used to convert an image from one color
# space to another.
#
# cv2.COLOR_BGR2GRAY:
#     Converts the image from BGR (Blue, Green, Red)
#     into a single-channel grayscale image.
#
# Grayscale pixel values:
#     0   = black
#     255 = white
#
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


# ============================================================
# STEP 3: DEFINE LOWER AND UPPER THRESHOLDS
# ============================================================

# Canny uses TWO thresholds:
#
# lower = lower threshold
# upper = upper threshold
#
# These thresholds are used to classify pixels according
# to their gradient strength.
#
# If gradient < lower:
#     Pixel is considered NON-EDGE.
#
# If lower <= gradient < upper:
#     Pixel is considered a WEAK EDGE candidate.
#
# If gradient >= upper:
#     Pixel is considered a STRONG EDGE.
#
lower = 50
upper = 150


# ============================================================
# STEP 4: CALCULATE HORIZONTAL AND VERTICAL GRADIENTS
# ============================================================

# Sobel operator is used to calculate how much intensity
# changes in the image.
#
# cv2.Sobel() parameters:
#
# gray:
#     Input grayscale image.
#
# cv2.CV_64F:
#     Output data type.
#     64-bit floating point is used because gradient values
#     can be negative and can be larger than 255.
#
# dx = 1, dy = 0:
#     Calculate gradient in the X direction.
#     This detects changes from LEFT to RIGHT.
#
# ksize=3:
#     Size of the Sobel kernel.
#     A 3x3 kernel is commonly used.
#
# gx = horizontal gradient
#
gx = cv2.Sobel(
    gray,
    cv2.CV_64F,
    1,
    0,
    ksize=3
)


# Calculate gradient in the Y direction.
#
# dx = 0, dy = 1:
#     Calculate gradient in the Y direction.
#     This detects changes from TOP to BOTTOM.
#
# gy = vertical gradient
#
gy = cv2.Sobel(
    gray,
    cv2.CV_64F,
    0,
    1,
    ksize=3
)


# ============================================================
# STEP 5: CALCULATE GRADIENT MAGNITUDE
# ============================================================

# Gradient magnitude tells us the TOTAL strength of the
# intensity change at each pixel.
#
# Conceptually:
#
#       magnitude = sqrt(gx^2 + gy^2)
#
# A large magnitude means:
#     Strong intensity change -> strong edge
#
# A small magnitude means:
#     Small intensity change -> weak/no edge
#
# cv2.magnitude() calculates:
#     sqrt(gx^2 + gy^2)
#
# It expects floating-point arrays, so gx and gy are
# converted to float32.
#
magnitude = cv2.magnitude(
    gx.astype(np.float32),
    gy.astype(np.float32)
)


# ============================================================
# STEP 6: IDENTIFY STRONG EDGES
# ============================================================

# A pixel is classified as a STRONG EDGE when its gradient
# magnitude is greater than or equal to the upper threshold.
#
# Example:
#
# upper = 150
#
# magnitude = 200
#     200 >= 150 -> STRONG EDGE
#
# magnitude = 100
#     100 >= 150 -> NOT strong
#
strong = magnitude >= upper


# ============================================================
# STEP 7: IDENTIFY WEAK EDGES
# ============================================================

# A pixel is classified as a WEAK EDGE candidate when:
#
#     lower <= magnitude < upper
#
# For:
#
# lower = 50
# upper = 150
#
# Example:
#
# magnitude = 100
#     50 <= 100 < 150 -> WEAK
#
# magnitude = 200
#     200 >= 150 -> STRONG
#
# magnitude = 20
#     20 < 50 -> NOT an edge candidate
#
weak = (
    (magnitude >= lower) &
    (magnitude < upper)
)


# ============================================================
# STEP 8: CREATE IMAGES FOR STRONG AND WEAK EDGES
# ============================================================

# np.zeros_like(gray) creates a black image having exactly
# the same dimensions as the grayscale image.
#
# Initially every pixel is 0 (black).
#
strong_edges = np.zeros_like(gray)
weak_edges = np.zeros_like(gray)


# Wherever the 'strong' condition is True, put 255.
#
# 255 = white
#
# Therefore:
#     Strong edge pixels -> white
#     Everything else   -> black
#
strong_edges[strong] = 255


# Similarly, wherever the 'weak' condition is True,
# put 255.
#
# Therefore:
#     Weak edge pixels -> white
#     Everything else -> black
#
weak_edges[weak] = 255


# ============================================================
# STEP 9: RUN ACTUAL CANNY EDGE DETECTION
# ============================================================

# cv2.Canny() performs the complete Canny edge detection
# algorithm.
#
# Parameters:
#
# gray:
#     Input grayscale image.
#
# lower:
#     Lower threshold.
#
# upper:
#     Upper threshold.
#
# Canny internally performs several steps:
#
# 1. Noise reduction
# 2. Gradient calculation
# 3. Non-maximum suppression
# 4. Double thresholding
# 5. Hysteresis
#
# The final result contains:
#     White = final detected edges
#     Black = non-edge pixels
#
canny_edges = cv2.Canny(
    gray,
    lower,
    upper
)


# ============================================================
# IMPORTANT:
#
# The 'weak_edges' image above is NOT exactly the final
# weak-edge result inside Canny.
#
# It simply shows pixels whose gradient magnitude lies
# between lower and upper thresholds.
#
# Canny uses HYSTERESIS to decide whether those weak pixels
# should actually be kept or removed.
#
# RULE:
#
# Weak edge connected to a strong edge
#       -> KEEP
#
# Weak edge NOT connected to a strong edge
#       -> REMOVE
#
# This is why some weak edges disappear from the final
# Canny output.
# ============================================================


# ============================================================
# STEP 10: DISPLAY ORIGINAL IMAGE
# ============================================================

plt.figure(figsize=(15, 8))


# subplot(rows, columns, position)
#
# 2 rows
# 3 columns
# position 1
#
plt.subplot(2, 3, 1)

# OpenCV loads image in BGR format, while matplotlib
# expects RGB format.
#
# Therefore BGR is converted to RGB before displaying.
plt.imshow(
    cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
)

plt.title('Original Image')
plt.axis('off')


# ============================================================
# STEP 11: DISPLAY GRAYSCALE IMAGE
# ============================================================

plt.subplot(2, 3, 2)

# cmap='gray' tells matplotlib to display the image
# using a grayscale colormap.
plt.imshow(
    gray,
    cmap='gray'
)

plt.title('Grayscale Image')
plt.axis('off')


# ============================================================
# STEP 12: DISPLAY STRONG EDGES
# ============================================================

plt.subplot(2, 3, 3)

plt.imshow(
    strong_edges,
    cmap='gray'
)

plt.title(
    f'Strong Edges (>= {upper})'
)

plt.axis('off')


# ============================================================
# STEP 13: DISPLAY WEAK EDGES
# ============================================================

plt.subplot(2, 3, 4)

plt.imshow(
    weak_edges,
    cmap='gray'
)

plt.title(
    f'Weak Edges ({lower} <= gradient < {upper})'
)

plt.axis('off')


# ============================================================
# STEP 14: DISPLAY FINAL CANNY EDGES
# ============================================================

plt.subplot(2, 3, 5)

plt.imshow(
    canny_edges,
    cmap='gray'
)

plt.title(
    f'Canny Edges ({lower}, {upper})'
)

plt.axis('off')


# ============================================================
# STEP 15: SHOW ALL PLOTS
# ============================================================

plt.tight_layout()
plt.show()


# ============================================================
# MISSING AND UNWANTED EDGES
# ============================================================
#
# IMPORTANT:
#
# We CANNOT identify "missing" and "unwanted" edges from
# Canny output alone.
#
# We need a GROUND TRUTH image (or expected edge image)
# telling us which edges SHOULD be present.
#
# Suppose:
#
# ground_truth = correct/expected edge image
#
# White pixels = edges that SHOULD be detected.
#
# Then we can compare the ground truth with Canny output.
# ============================================================


# ------------------------------------------------------------
# STEP 16: LOAD GROUND TRUTH EDGE IMAGE
# ------------------------------------------------------------

# Replace 'ground_truth.png' with the actual path of your
# ground-truth edge image.
#
# The ground-truth image should ideally be a binary image:
#
#     255 = desired/true edge
#     0   = background
#
ground_truth = cv2.imread(
    'ground_truth.png',
    cv2.IMREAD_GRAYSCALE
)


# ------------------------------------------------------------
# STEP 17: CONVERT GROUND TRUTH INTO BINARY FORM
# ------------------------------------------------------------

# Threshold the ground-truth image so that it contains
# only 0 and 255.
#
# Pixels >= 127 become 255.
# Pixels < 127 become 0.
#
_, ground_truth = cv2.threshold(
    ground_truth,
    127,
    255,
    cv2.THRESH_BINARY
)


# ============================================================
# STEP 18: FIND MISSING EDGES
# ============================================================

# MISSING EDGE means:
#
#     The edge SHOULD be present according to ground truth
#     BUT Canny DID NOT detect it.
#
# Formula:
#
#     Missing = Ground Truth AND NOT Canny
#
# cv2.bitwise_not(canny_edges):
#     Changes:
#         255 -> 0
#         0   -> 255
#
# cv2.bitwise_and():
#     Keeps pixels that are white in BOTH images.
#
missing = cv2.bitwise_and(
    ground_truth,
    cv2.bitwise_not(canny_edges)
)


# ============================================================
# STEP 19: FIND UNWANTED EDGES
# ============================================================

# UNWANTED EDGE means:
#
#     Canny detected an edge
#     BUT the edge is NOT present in the ground truth.
#
# Formula:
#
#     Unwanted = Canny AND NOT Ground Truth
#
unwanted = cv2.bitwise_and(
    canny_edges,
    cv2.bitwise_not(ground_truth)
)


# ============================================================
# STEP 20: DISPLAY MISSING AND UNWANTED EDGES
# ============================================================

plt.figure(figsize=(12, 5))


# Display ground truth
plt.subplot(1, 3, 1)

plt.imshow(
    ground_truth,
    cmap='gray'
)

plt.title('Ground Truth')
plt.axis('off')


# Display missing edges
plt.subplot(1, 3, 2)

plt.imshow(
    missing,
    cmap='gray'
)

plt.title('Missing Edges')
plt.axis('off')


# Display unwanted edges
plt.subplot(1, 3, 3)

plt.imshow(
    unwanted,
    cmap='gray'
)

plt.title('Unwanted Edges')
plt.axis('off')


plt.tight_layout()
plt.show()


# ============================================================
# FINAL SUMMARY
# ============================================================
#
# STRONG EDGE:
#     gradient >= upper threshold
#
# WEAK EDGE:
#     lower <= gradient < upper
#
# NON-EDGE:
#     gradient < lower
#
# MISSING EDGE:
#     Should be present in ground truth
#     BUT not detected by Canny
#
# UNWANTED EDGE:
#     Detected by Canny
#     BUT not present in ground truth
#
# CANNY HYSTERESIS:
#
#     Strong edge ----------------> KEEP
#
#     Weak edge + connected to
#     strong edge ----------------> KEEP
#
#     Weak edge + NOT connected
#     to strong edge --------------> REMOVE
#
# ============================================================

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 8: WATERSHED DISTANCE-TRANSFORM PARAMETER STUDY
# ============================================================

# ------------------------------------------------------------
# 1. LOAD IMAGE
# ------------------------------------------------------------
image = cv2.imread('coins.jpg')

# Keep an original copy because watershed modifies the image
original = image.copy()


# ------------------------------------------------------------
# 2. CONVERT IMAGE TO GRAYSCALE
# ------------------------------------------------------------
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


# ------------------------------------------------------------
# 3. APPLY OTSU THRESHOLDING
# ------------------------------------------------------------
# THRESH_BINARY_INV:
#     Converts the image into binary form and inverts it.
#
# THRESH_OTSU:
#     Automatically finds the best threshold value.
#
# 0:
#     Initial threshold value. Otsu calculates the actual value,
#     so we don't need to manually provide it.
#
# _:
#     We ignore the threshold value returned by cv2.threshold().
# ------------------------------------------------------------
_, thresh = cv2.threshold(
    gray,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)


# ------------------------------------------------------------
# 4. CREATE MORPHOLOGICAL KERNEL
# ------------------------------------------------------------
# A 3x3 kernel is used for morphological operations.
#
# np.ones():
#     Creates a matrix filled with 1s.
#
# np.uint8:
#     Converts the kernel into unsigned 8-bit integer format.
# ------------------------------------------------------------
kernel = np.ones((3, 3), np.uint8)


# ------------------------------------------------------------
# 5. MORPHOLOGICAL OPENING
# ------------------------------------------------------------
# Opening = Erosion followed by Dilation.
#
# It removes small noise from the binary image.
# iterations=2 means the operation is performed twice.
# ------------------------------------------------------------
opening = cv2.morphologyEx(
    thresh,
    cv2.MORPH_OPEN,
    kernel,
    iterations=2
)


# ------------------------------------------------------------
# 6. FIND SURE BACKGROUND
# ------------------------------------------------------------
# Dilation expands the white regions.
#
# This gives us an area that is very likely to be background.
# ------------------------------------------------------------
sure_bg = cv2.dilate(
    opening,
    kernel,
    iterations=3
)


# ------------------------------------------------------------
# 7. DISTANCE TRANSFORM
# ------------------------------------------------------------
# Distance transform tells us how far each foreground pixel
# is from the nearest background pixel.
#
# DIST_L2:
#     Uses Euclidean distance.
#
# 5:
#     Mask size used for calculating the distance.
# ------------------------------------------------------------
dist_transform = cv2.distanceTransform(
    opening,
    cv2.DIST_L2,
    5
)


# ------------------------------------------------------------
# 8. DIFFERENT DISTANCE-TRANSFORM THRESHOLDS
# ------------------------------------------------------------
# These ratios will be tested one by one.
#
# 0.10 = 10% of maximum distance
# 0.20 = 20%
# 0.30 = 30%
# 0.40 = 40%
#
# Higher threshold means we select only the deeper/central
# parts of the objects as sure foreground.
# ------------------------------------------------------------
distance_thresholds = [0.10, 0.20, 0.30, 0.40]


# ------------------------------------------------------------
# LISTS TO STORE RESULTS
# ------------------------------------------------------------

results = []

# This list stores the visual quality.
# We will fill these after looking at the output images.
visual_quality = [
    "Check image",
    "Check image",
    "Check image",
    "Check image"
]


# ============================================================
# 9. RUN WATERSHED FOR EVERY DISTANCE THRESHOLD
# ============================================================

for i in range(len(distance_thresholds)):

    # Get current threshold ratio
    ratio = distance_thresholds[i]


    # --------------------------------------------------------
    # CREATE SURE FOREGROUND
    # --------------------------------------------------------
    # ratio * max distance gives the actual threshold.
    #
    # Example:
    # If max distance = 100
    # and ratio = 0.20
    #
    # threshold = 20
    #
    # Only pixels having distance >= 20 will be selected.
    # --------------------------------------------------------
    _, sure_fg = cv2.threshold(
        dist_transform,
        ratio * dist_transform.max(),
        255,
        0
    )


    # Convert foreground image to uint8
    sure_fg = np.uint8(sure_fg)


    # --------------------------------------------------------
    # FIND UNKNOWN REGION
    # --------------------------------------------------------
    # Background - Foreground = Unknown region
    #
    # This region is neither definitely background nor
    # definitely foreground.
    # --------------------------------------------------------
    unknown = cv2.subtract(
        sure_bg,
        sure_fg
    )


    # --------------------------------------------------------
    # FIND CONNECTED COMPONENTS
    # --------------------------------------------------------
    # connectedComponents() labels every connected foreground
    # object with a different number.
    #
    # num_labels:
    #     Total number of labels INCLUDING background.
    #
    # markers:
    #     Image containing the labels.
    # --------------------------------------------------------
    num_labels, markers = cv2.connectedComponents(
        sure_fg
    )


    # --------------------------------------------------------
    # NUMBER OF FOREGROUND MARKERS
    # --------------------------------------------------------
    # Label 0 is the background.
    #
    # Therefore:
    #
    # foreground markers = total labels - 1
    # --------------------------------------------------------
    foreground_markers = num_labels - 1


    # --------------------------------------------------------
    # SHIFT MARKER LABELS
    # --------------------------------------------------------
    # Add 1 so that background becomes label 1.
    #
    # This allows us to reserve:
    #
    # 0  -> unknown region
    # 1  -> background
    # 2,3,4,... -> foreground regions
    # --------------------------------------------------------
    markers = markers + 1


    # --------------------------------------------------------
    # MARK UNKNOWN REGION AS 0
    # --------------------------------------------------------
    markers[unknown == 255] = 0


    # --------------------------------------------------------
    # CREATE A FRESH COPY FOR WATERSHED
    # --------------------------------------------------------
    watershed_image = original.copy()


    # --------------------------------------------------------
    # APPLY WATERSHED
    # --------------------------------------------------------
    # Watershed uses the markers to separate touching objects.
    #
    # Boundary pixels are assigned the value -1.
    # --------------------------------------------------------
    cv2.watershed(
        watershed_image,
        markers
    )


    # --------------------------------------------------------
    # COUNT FINAL SEPARATED WATERSHED REGIONS
    # --------------------------------------------------------
    # After watershed:
    #
    # -1 = boundary
    #  1 = background
    #  2,3,4,... = separated foreground regions
    #
    # We only select labels greater than 1.
    # np.unique() gives all unique labels.
    # --------------------------------------------------------
    unique_labels = np.unique(
        markers[markers > 1]
    )

    separated_regions = len(unique_labels)


    # --------------------------------------------------------
    # COLOR WATERSHED BOUNDARIES RED
    # --------------------------------------------------------
    # OpenCV uses BGR format.
    #
    # [0, 0, 255] = RED in BGR.
    # --------------------------------------------------------
    watershed_image[
        markers == -1
    ] = [0, 0, 255]


    # --------------------------------------------------------
    # STORE ALL IMPORTANT RESULTS
    # --------------------------------------------------------
    results.append(
        (
            ratio,
            foreground_markers,
            separated_regions,
            watershed_image
        )
    )


# ============================================================
# 10. DISPLAY ALL SEGMENTATION RESULTS
# ============================================================

plt.figure(figsize=(16, 12))


for i in range(len(results)):

    # Extract information from results
    ratio = results[i][0]
    foreground_markers = results[i][1]
    separated_regions = results[i][2]
    result = results[i][3]


    # Create subplot
    plt.subplot(2, 2, i + 1)


    # OpenCV uses BGR, while Matplotlib uses RGB.
    plt.imshow(
        cv2.cvtColor(
            result,
            cv2.COLOR_BGR2RGB
        )
    )


    # Display threshold and region count
    plt.title(
        f'Threshold = {ratio}\n'
        f'Markers = {foreground_markers}, '
        f'Regions = {separated_regions}'
    )


    plt.axis('off')


plt.tight_layout()
plt.show()


# ============================================================
# 11. CREATE FINAL TABLE
# ============================================================

print("\n")
print("=" * 90)
print("WATERSHED PARAMETER STUDY RESULTS")
print("=" * 90)

print(
    f"{'Distance Threshold':<22}"
    f"{'Foreground Markers':<22}"
    f"{'Separated Regions':<22}"
    f"{'Visual Quality':<20}"
)

print("-" * 90)


for i in range(len(results)):

    ratio = results[i][0]
    foreground_markers = results[i][1]
    separated_regions = results[i][2]

    print(
        f"{ratio:<22}"
        f"{foreground_markers:<22}"
        f"{separated_regions:<22}"
        f"{visual_quality[i]:<20}"
    )

print("=" * 90)

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 9: K-MEANS IMAGE SEGMENTATION
# ============================================================


# ------------------------------------------------------------
# 1. LOAD COLOR IMAGE
# ------------------------------------------------------------

image = cv2.imread(
    'wildlife.jpg'
)


# ------------------------------------------------------------
# 2. RESHAPE IMAGE INTO PIXEL FEATURE VECTORS
# ------------------------------------------------------------
# Original image shape:
#
#     height x width x 3
#
# Each pixel contains:
#
#     B, G, R
#
# We convert the image into rows of pixels:
#
#     Pixel 1 -> B G R
#     Pixel 2 -> B G R
#     Pixel 3 -> B G R
#
# -1 means NumPy automatically calculates the
# required number of rows.
# ------------------------------------------------------------

pixel_values = image.reshape(
    (-1, 3)
)


# ------------------------------------------------------------
# 3. CONVERT PIXELS TO FLOAT32
# ------------------------------------------------------------
# cv2.kmeans() requires floating-point data.
# ------------------------------------------------------------

pixel_values = np.float32(
    pixel_values
)


# ------------------------------------------------------------
# 4. DEFINE STOPPING CRITERIA
# ------------------------------------------------------------
# EPS:
# Stop when the centroid movement becomes very small.
#
# MAX_ITER:
# Maximum number of iterations = 100
#
# 0.2:
# Epsilon value.
# ------------------------------------------------------------

criteria = (
    cv2.TERM_CRITERIA_EPS +
    cv2.TERM_CRITERIA_MAX_ITER,
    100,
    0.2
)


# ------------------------------------------------------------
# 5. TEST DIFFERENT VALUES OF K
# ------------------------------------------------------------

K_values = [
    2,
    4,
    6
]


# ------------------------------------------------------------
# LIST TO STORE ALL RESULTS
# ------------------------------------------------------------

results = []


# ============================================================
# 6. APPLY K-MEANS FOR EACH K
# ============================================================

for K in K_values:

    # --------------------------------------------------------
    # APPLY K-MEANS
    # --------------------------------------------------------

    compactness, labels, centers = cv2.kmeans(
        pixel_values,
        K,
        None,
        criteria,
        10,
        cv2.KMEANS_RANDOM_CENTERS
    )


    # --------------------------------------------------------
    # CONVERT CLUSTER CENTERS TO uint8
    # --------------------------------------------------------

    centers = np.uint8(
        centers
    )


    # --------------------------------------------------------
    # REPLACE EACH PIXEL WITH ITS CLUSTER CENTER
    # --------------------------------------------------------
    # Every pixel is replaced by the representative
    # color of its cluster.
    # --------------------------------------------------------

    segmented_image = centers[
        labels.flatten()
    ]


    # --------------------------------------------------------
    # RESHAPE BACK TO ORIGINAL IMAGE SIZE
    # --------------------------------------------------------

    segmented_image = segmented_image.reshape(
        image.shape
    )


    # ========================================================
    # 7. VISUAL DIFFERENCE FROM ORIGINAL
    # ========================================================
    #
    # We calculate the Mean Absolute Difference (MAD).
    #
    # A LOW value means:
    #     segmented image is visually closer to original.
    #
    # A HIGH value means:
    #     more difference from original.
    #
    # We convert both images to float32 so that subtraction
    # does not cause uint8 problems.
    # ========================================================

    difference = np.abs(
        image.astype(np.float32)
        -
        segmented_image.astype(np.float32)
    )

    visual_difference = np.mean(
        difference
    )


    # ========================================================
    # 8. AMOUNT OF COLOR SIMPLIFICATION
    # ========================================================
    #
    # Original image may contain thousands of different
    # RGB/BGR color combinations.
    #
    # K-means reduces them to approximately K representative
    # colors.
    #
    # We count the unique colors before and after segmentation.
    # ========================================================

    original_colors = len(
        np.unique(
            image.reshape(-1, 3),
            axis=0
        )
    )

    segmented_colors = len(
        np.unique(
            segmented_image.reshape(-1, 3),
            axis=0
        )
    )


    # Calculate percentage of color reduction.
    color_simplification = (
        (original_colors - segmented_colors)
        / original_colors
    ) * 100


    # ========================================================
    # 9. FIND MAJOR REGIONS REPRESENTED BY CLUSTERS
    # ========================================================
    #
    # labels tells us which cluster each pixel belongs to.
    #
    # We count how many pixels belong to each cluster.
    #
    # A cluster containing many pixels represents a large
    # region of the image.
    # ========================================================

    cluster_counts = []

    for cluster_number in range(K):

        count = np.sum(
            labels.flatten() == cluster_number
        )

        cluster_counts.append(
            count
        )


    # --------------------------------------------------------
    # SORT CLUSTERS FROM LARGEST TO SMALLEST
    # --------------------------------------------------------

    sorted_clusters = np.argsort(
        cluster_counts
    )[::-1]


    # --------------------------------------------------------
    # FIND MAJOR CLUSTERS
    # --------------------------------------------------------
    # Here we consider clusters containing at least 5%
    # of the total pixels as major regions.
    # --------------------------------------------------------

    total_pixels = image.shape[0] * image.shape[1]

    major_regions = []

    for cluster_number in sorted_clusters:

        percentage = (
            cluster_counts[cluster_number]
            / total_pixels
        ) * 100

        if percentage >= 5:

            major_regions.append(
                f'Cluster {cluster_number}: '
                f'{percentage:.1f}%'
            )


    # If no cluster reaches 5%
    if len(major_regions) == 0:

        major_regions.append(
            'No major cluster'
        )


    # ========================================================
    # 10. STORE ALL RESULTS
    # ========================================================

    results.append(
        (
            K,
            visual_difference,
            original_colors,
            segmented_colors,
            color_simplification,
            major_regions,
            segmented_image
        )
    )


# ============================================================
# 11. DISPLAY ORIGINAL + SEGMENTED IMAGES
# ============================================================

plt.figure(
    figsize=(16, 5)
)


# ------------------------------------------------------------
# ORIGINAL IMAGE
# ------------------------------------------------------------

plt.subplot(
    1,
    4,
    1
)

plt.imshow(
    cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
)

plt.title(
    'Original'
)

plt.axis(
    'off'
)


# ------------------------------------------------------------
# DISPLAY K = 2, 4, 6
# ------------------------------------------------------------

for i in range(
    len(results)
):

    K = results[i][0]

    result = results[i][6]


    plt.subplot(
        1,
        4,
        i + 2
    )


    plt.imshow(
        cv2.cvtColor(
            result,
            cv2.COLOR_BGR2RGB
        )
    )


    plt.title(
        f'K = {K}'
    )


    plt.axis(
        'off'
    )


plt.tight_layout()

plt.show()


# ============================================================
# 12. PRINT FINAL RESULTS TABLE
# ============================================================

print("\n")
print("=" * 120)

print(
    "K-MEANS IMAGE SEGMENTATION RESULTS"
)

print("=" * 120)


print(
    f"{'K':<10}"
    f"{'Visual Difference':<22}"
    f"{'Color Simplification':<25}"
    f"{'Major Regions':<50}"
)

print("-" * 120)


for i in range(
    len(results)
):

    K = results[i][0]

    visual_difference = results[i][1]

    color_simplification = results[i][4]

    major_regions = ", ".join(
        results[i][5]
    )


    print(
        f"{K:<10}"
        f"{visual_difference:<22.2f}"
        f"{color_simplification:<25.2f}%"
        f"{major_regions:<50}"
    )


print("=" * 120)


# ============================================================
# 13. SAVE EACH SEGMENTED IMAGE
# ============================================================

for i in range(
    len(results)
):

    K = results[i][0]

    result = results[i][6]


    cv2.imwrite(
        f'task9_K{K}.png',
        result
    )

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 10: SEGMENTATION SYSTEM FOR AN UNKNOWN IMAGE
# ============================================================
#
# IMAGE:
# Unevenly illuminated document
#
# SELECTED METHODS:
# 1. Global Thresholding
# 2. Adaptive Thresholding
# 3. Otsu's Thresholding
#
# For every method we will identify:
# 1. Assumption
# 2. Successfully identified region/object
# 3. Incorrectly segmented part
# 4. Most important parameter
#
# We will also create:
# - Final visual comparison
# - Technical conclusion
# - Comparison table
# ============================================================


# ============================================================
# 1. LOAD IMAGE
# ============================================================

image = cv2.imread(
    'document.jpg',
    cv2.IMREAD_GRAYSCALE
)


# Check whether image was loaded successfully
if image is None:

    print("Error: document.jpg could not be loaded.")

    exit()


# ============================================================
# METHOD 1: GLOBAL THRESHOLDING
# ============================================================

# ------------------------------------------------------------
# ASSUMPTION:
# ------------------------------------------------------------
# Global thresholding assumes that one threshold value
# can separate the foreground (text/document content)
# from the background over the entire image.
#
# MAIN PARAMETER:
# 128 = threshold value
#
# Pixels:
# > 128  -> white
# <= 128 -> black
# ------------------------------------------------------------

global_threshold = 128

_, global_result = cv2.threshold(
    image,
    global_threshold,
    255,
    cv2.THRESH_BINARY
)


# ============================================================
# METHOD 2: ADAPTIVE THRESHOLDING
# ============================================================

# ------------------------------------------------------------
# ASSUMPTION:
# ------------------------------------------------------------
# Adaptive thresholding assumes that different local areas
# of the image may have different illumination conditions.
#
# Therefore, instead of using ONE threshold for the whole
# image, it calculates a different threshold for local
# neighborhoods.
#
# This is particularly useful for uneven illumination.
#
# PARAMETERS:
#
# 11 -> block size
#      Size of the local neighborhood.
#
# 5 -> C
#      Value subtracted from the calculated local threshold.
# ------------------------------------------------------------

adaptive_block_size = 11
adaptive_C = 5

adaptive_result = cv2.adaptiveThreshold(
    image,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    adaptive_block_size,
    adaptive_C
)


# ============================================================
# METHOD 3: OTSU'S THRESHOLDING
# ============================================================

# ------------------------------------------------------------
# ASSUMPTION:
# ------------------------------------------------------------
# Otsu assumes that the image can be separated into two
# major intensity classes:
#
#     Foreground
#          +
#     Background
#
# It automatically finds the threshold that gives the best
# separation between these two intensity groups.
#
# MAIN PARAMETER:
# Otsu automatically calculates the threshold.
#
# The initial threshold is set to 0 because Otsu will
# calculate the actual value.
# ------------------------------------------------------------

otsu_threshold, otsu_result = cv2.threshold(
    image,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)


# ============================================================
# 4. CALCULATE BASIC OBSERVATIONS
# ============================================================
#
# We calculate the percentage of white pixels in each
# segmentation result.
#
# This helps us understand how much of the image each method
# classified as foreground/white.
# ============================================================

total_pixels = image.shape[0] * image.shape[1]


# ------------------------------------------------------------
# Global white pixels
# ------------------------------------------------------------

global_white_pixels = np.sum(
    global_result == 255
)

global_white_percentage = (
    global_white_pixels / total_pixels
) * 100


# ------------------------------------------------------------
# Adaptive white pixels
# ------------------------------------------------------------

adaptive_white_pixels = np.sum(
    adaptive_result == 255
)

adaptive_white_percentage = (
    adaptive_white_pixels / total_pixels
) * 100


# ------------------------------------------------------------
# Otsu white pixels
# ------------------------------------------------------------

otsu_white_pixels = np.sum(
    otsu_result == 255
)

otsu_white_percentage = (
    otsu_white_pixels / total_pixels
) * 100


# ============================================================
# 5. PRINT METHOD ANALYSIS
# ============================================================

print("\n")
print("=" * 100)
print("TASK 10 - SEGMENTATION METHOD ANALYSIS")
print("=" * 100)


# ============================================================
# METHOD 1 ANALYSIS
# ============================================================

print("\nMETHOD 1: GLOBAL THRESHOLDING")
print("-" * 60)

print(
    "1. Assumption:"
)

print(
    "A single threshold can separate foreground/text "
    "from the background across the entire image."
)

print(
    "\n2. Successfully identifies:"
)

print(
    "Dark text/foreground regions that have sufficient "
    "contrast with the document background."
)

print(
    "\n3. Incorrectly segments:"
)

print(
    "Dark shadowed background areas may be classified as "
    "foreground, while faint text in bright areas may be missed."
)

print(
    "\n4. Most influential parameter:"
)

print(
    "Threshold value =", global_threshold
)

print(
    "\nObserved white/foreground area:",
    f"{global_white_percentage:.2f}%"
)


# ============================================================
# METHOD 2 ANALYSIS
# ============================================================

print("\nMETHOD 2: ADAPTIVE THRESHOLDING")
print("-" * 60)

print(
    "1. Assumption:"
)

print(
    "Different parts of the image may have different "
    "illumination conditions, so local thresholds are needed."
)

print(
    "\n2. Successfully identifies:"
)

print(
    "Text and document details even when different areas "
    "of the document have different brightness levels."
)

print(
    "\n3. Incorrectly segments:"
)

print(
    "Small shadows, paper texture, noise, or local intensity "
    "changes may sometimes be incorrectly classified as foreground."
)

print(
    "\n4. Most influential parameters:"
)

print(
    "Block size =",
    adaptive_block_size
)

print(
    "C =",
    adaptive_C
)

print(
    "\nObserved white/foreground area:",
    f"{adaptive_white_percentage:.2f}%"
)


# ============================================================
# METHOD 3 ANALYSIS
# ============================================================

print("\nMETHOD 3: OTSU THRESHOLDING")
print("-" * 60)

print(
    "1. Assumption:"
)

print(
    "The image intensity histogram can be separated into "
    "two major classes: foreground and background."
)

print(
    "\n2. Successfully identifies:"
)

print(
    "Large-scale foreground and background regions when "
    "their intensity distributions are reasonably distinct."
)

print(
    "\n3. Incorrectly segments:"
)

print(
    "Unevenly illuminated areas may be incorrectly classified "
    "because Otsu uses one global threshold for the whole image."
)

print(
    "\n4. Most influential parameter:"
)

print(
    "Automatically selected Otsu threshold =",
    otsu_threshold
)

print(
    "\nObserved white/foreground area:",
    f"{otsu_white_percentage:.2f}%"
)


# ============================================================
# 6. FINAL VISUAL COMPARISON
# ============================================================

plt.figure(
    figsize=(16, 5)
)


# ------------------------------------------------------------
# ORIGINAL
# ------------------------------------------------------------

plt.subplot(
    1,
    4,
    1
)

plt.imshow(
    image,
    cmap='gray'
)

plt.title(
    'Original'
)

plt.axis(
    'off'
)


# ------------------------------------------------------------
# GLOBAL THRESHOLD
# ------------------------------------------------------------

plt.subplot(
    1,
    4,
    2
)

plt.imshow(
    global_result,
    cmap='gray'
)

plt.title(
    f'Global\nT = {global_threshold}'
)

plt.axis(
    'off'
)


# ------------------------------------------------------------
# ADAPTIVE THRESHOLD
# ------------------------------------------------------------

plt.subplot(
    1,
    4,
    3
)

plt.imshow(
    adaptive_result,
    cmap='gray'
)

plt.title(
    f'Adaptive\nBlock={adaptive_block_size}, C={adaptive_C}'
)

plt.axis(
    'off'
)


# ------------------------------------------------------------
# OTSU
# ------------------------------------------------------------

plt.subplot(
    1,
    4,
    4
)

plt.imshow(
    otsu_result,
    cmap='gray'
)

plt.title(
    f'Otsu\nT = {otsu_threshold:.0f}'
)

plt.axis(
    'off'
)


plt.tight_layout()

plt.show()


# ============================================================
# 7. FINAL COMPARISON TABLE
# ============================================================

print("\n")
print("=" * 120)
print("FINAL COMPARISON")
print("=" * 120)

print(
    f"{'Method':<25}"
    f"{'Assumption':<30}"
    f"{'Successful Region':<30}"
    f"{'Main Parameter':<30}"
)

print("-" * 120)

print(
    f"{'Global Threshold':<25}"
    f"{'One threshold for image':<30}"
    f"{'High-contrast text':<30}"
    f"{'Threshold = 128':<30}"
)

print(
    f"{'Adaptive Threshold':<25}"
    f"{'Local illumination varies':<30}"
    f"{'Text under uneven lighting':<30}"
    f"{'Block=11, C=5':<30}"
)

print(
    f"{'Otsu':<25}"
    f"{'Two intensity classes':<30}"
    f"{'Distinct foreground/background':<30}"
    f"{'Auto T = ' + str(round(otsu_threshold, 2)):<30}"
)

print("=" * 120)


# ============================================================
# 8. TECHNICAL CONCLUSION
# ============================================================
#
# Because this image is specifically an unevenly illuminated
# document, Adaptive Thresholding is expected to perform best.
#
# HOWEVER, the task says that the conclusion must be based
# on observed results.
#
# Therefore, we display the conclusion after comparing the
# actual results.
# ============================================================

print("\n")
print("=" * 100)
print("TECHNICAL CONCLUSION")
print("=" * 100)

print(
    """
For this image, Adaptive Thresholding is the most meaningful
segmentation approach if the displayed result preserves the
document text across both bright and shadowed areas.

The reason is that the document has uneven illumination.
Global Thresholding and Otsu use one threshold for the entire
image, so a threshold suitable for one area may not be suitable
for another area.

Adaptive Thresholding calculates local thresholds, allowing
text to be detected even when the brightness of the document
changes from one region to another.

The final choice is based on visual comparison of the three
segmentation results rather than simply selecting an algorithm
in advance.
"""
)


# ============================================================
# 9. SAVE RESULTS
# ============================================================

cv2.imwrite(
    'task10_global.png',
    global_result
)

cv2.imwrite(
    'task10_adaptive.png',
    adaptive_result
)

cv2.imwrite(
    'task10_otsu.png',
    otsu_result
)


print("\nSegmentation results saved successfully.")

In [ ]:
cv2.imread(filename, flags)

In [ ]:
cv2.IMREAD_COLOR
cv2.IMREAD_GRAYSCALE
cv2.IMREAD_UNCHANGED

In [ ]:
cv2.threshold(
    src,
    thresh,
    maxval,
    type
)

cv2.THRESH_BINARY
cv2.THRESH_BINARY_INV
cv2.THRESH_TRUNC
cv2.THRESH_TOZERO
cv2.THRESH_TOZERO_INV
cv2.THRESH_OTSU
cv2.THRESH_TRIANGLE

In [ ]:
cv2.adaptiveThreshold(
    src,
    maxValue,
    adaptiveMethod,
    thresholdType,
    blockSize,
    C
)

cv2.ADAPTIVE_THRESH_MEAN_C
cv2.ADAPTIVE_THRESH_GAUSSIAN_C

In [ ]:
cv2.cvtColor(src, code)

cv2.COLOR_BGR2GRAY
cv2.COLOR_BGR2RGB
cv2.COLOR_BGR2HSV
cv2.COLOR_RGB2GRAY
cv2.COLOR_GRAY2BGR

In [ ]:
cv2.inRange(
    src,
    lowerb,
    upperb
)

In [ ]:
cv2.Canny(
    image,
    threshold1,
    threshold2
)
# optional parameters
apertureSize=3
L2gradient=False

In [ ]:
cv2.morphologyEx(
    src,
    op,
    kernel,
    iterations=1
)
cv2.MORPH_OPEN
cv2.MORPH_CLOSE
cv2.MORPH_GRADIENT
cv2.MORPH_TOPHAT
cv2.MORPH_BLACKHAT

In [ ]:
cv2.dilate(
    src,
    kernel,
    iterations=1
)
cv2.distanceTransform(
    src,
    distanceType,
    maskSize
)
cv2.DIST_L1
cv2.DIST_L2
cv2.DIST_C
cv2.connectedComponents(
    image
)
cv2.watershed(
    image,
    markers
)
markers == -1
cv2.kmeans(
    data,
    K,
    bestLabels,
    criteria,
    attempts,
    flags
)

In [ ]:
# CODE 1 — GLOBAL THRESHOLDING
import cv2

# Read image in grayscale
image = cv2.imread(
    'image.jpg',
    cv2.IMREAD_GRAYSCALE
)

# Apply global binary thresholding
# 128 = threshold value
# 255 = maximum output value
# THRESH_BINARY = pixels above threshold become 255,
#                 remaining pixels become 0
_, binary_mask = cv2.threshold(
    image,
    128,
    255,
    cv2.THRESH_BINARY
)

# binary_mask contains the segmented binary image

In [ ]:
# CODE 2 — ADAPTIVE THRESHOLDING
import cv2

# Read image as grayscale
image = cv2.imread(
    'image.jpg',
    cv2.IMREAD_GRAYSCALE
)

# Apply adaptive thresholding
binary_mask = cv2.adaptiveThreshold(
    image,
    255,                              # Maximum output value
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,   # Gaussian local threshold
    cv2.THRESH_BINARY,                # Binary output
    11,                               # Block size: 11 x 11
    2                                 # Constant C
)

# Adaptive thresholding is useful for uneven lighting

In [ ]:
# CODE 3 — OTSU THRESHOLDING
import cv2

# Read image in grayscale
image = cv2.imread(
    'image.jpg',
    cv2.IMREAD_GRAYSCALE
)

# Otsu automatically finds an appropriate threshold
# Therefore thresh is given as 0
_, binary_mask = cv2.threshold(
    image,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Otsu works well when image intensity distribution
# is approximately bimodal



In [ ]:
# CODE 4 — COLOR-BASED THRESHOLDING
import cv2
import numpy as np

# Read color image
image = cv2.imread('image.jpg')

# Convert BGR image to HSV
hsv_image = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2HSV
)

# Lower HSV boundary
lower_bound = np.array([
    30,   # Lower Hue
    50,   # Lower Saturation
    50    # Lower Value
])

# Upper HSV boundary
upper_bound = np.array([
    60,   # Upper Hue
    255,  # Upper Saturation
    255   # Upper Value
])

# Select pixels whose HSV values fall
# inside the specified range
mask = cv2.inRange(
    hsv_image,
    lower_bound,
    upper_bound
)

# mask:
# 255 = selected color range
# 0   = outside color range

In [ ]:
# CODE 5 — CANNY / HYSTERESIS THRESHOLDING
import cv2

# Read image as grayscale
image = cv2.imread(
    'image.jpg',
    cv2.IMREAD_GRAYSCALE
)

# Apply Canny edge detection
# 100 = lower threshold
# 200 = upper threshold
edges = cv2.Canny(
    image,
    100,
    200
)

# Strong edges are detected using the high threshold.
# Weak edges are retained if connected to strong edges.
# CODE 6 — REGION GROWING
import cv2
import numpy as np
from matplotlib import pyplot as plt

# Load grayscale image
image = cv2.imread(
    'ents.jpg',
    cv2.IMREAD_GRAYSCALE
)

# Starting pixel for region growing
# Format: (row, column)
seed_point = (10, 10)


# Function for region growing
def region_growing(image, seed, threshold):

    # Create an empty mask having the same
    # shape as the input image
    mask = np.zeros_like(
        image,
        dtype=np.uint8
    )

    # Stack contains pixels that need to be checked
    stack = [seed]

    # Intensity of the initial seed pixel
    seed_intensity = image[seed]

    # Continue until there are no pixels left
    # in the stack
    while stack:

        # Remove the last pixel from the stack
        x, y = stack.pop()

        # Check whether pixel is outside image boundaries
        if x < 0 or x >= image.shape[0] or \
           y < 0 or y >= image.shape[1]:
            continue

        # Check whether pixel has not already
        # been included in the region
        if mask[x, y] == 0:

            # Check similarity with seed intensity
            if abs(
                int(image[x, y]) -
                int(seed_intensity)
            ) < threshold:

                # Add pixel to segmented region
                mask[x, y] = 255

                # Add 4-connected neighboring pixels
                stack.extend([
                    (x + 1, y),
                    (x - 1, y),
                    (x, y + 1),
                    (x, y - 1)
                ])

    # Return segmented region
    return mask


# Similarity threshold
threshold = 50

# Perform region growing
segmented_image = region_growing(
    image,
    seed_point,
    threshold
)

# Display segmented result
plt.imshow(
    segmented_image,
    cmap='gray'
)

plt.axis('off')
plt.show()


# CODE 7 — WATERSHED SEGMENTATION
import cv2
import numpy as np
from matplotlib import pyplot as plt

# Load original color image
image = cv2.imread('dog.jpeg')

# Convert BGR image to grayscale
gray = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2GRAY
)

# Threshold image using Otsu
# THRESH_BINARY_INV creates an inverted binary image
_, thresh = cv2.threshold(
    gray,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

# Create 5 x 5 kernel
kernel = np.ones(
    (5, 5),
    np.uint8
)

# Morphological opening:
# erosion followed by dilation
# Used to remove small noise
opening = cv2.morphologyEx(
    thresh,
    cv2.MORPH_OPEN,
    kernel,
    iterations=2
)

# Dilate to obtain sure background
sure_bg = cv2.dilate(
    opening,
    kernel,
    iterations=3
)

# Distance transform:
# calculates distance of foreground pixels
# from nearest background
dist_transform = cv2.distanceTransform(
    opening,
    cv2.DIST_L2,
    5
)

# Obtain sure foreground
_, sure_fg = cv2.threshold(
    dist_transform,
    0.2 * dist_transform.max(),
    255,
    0
)

# Convert foreground mask to uint8
sure_fg = np.uint8(sure_fg)

# Unknown region = sure background - sure foreground
unknown = cv2.subtract(
    sure_bg,
    sure_fg
)

# Label connected foreground components
_, markers = cv2.connectedComponents(
    sure_fg
)

# Shift marker labels by 1
markers = markers + 1

# Mark unknown region with 0
markers[unknown == 255] = 0

# Apply watershed segmentation
cv2.watershed(
    image,
    markers
)

# Watershed boundaries have marker value -1
# OpenCV uses BGR ordering
# [0, 0, 255] = red
image[markers == -1] = [
    0,
    0,
    255
]

# Display result
plt.imshow(
    cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis('off')
plt.show()


# CODE 8 — K-MEANS SEGMENTATION
import cv2
import numpy as np
from matplotlib import pyplot as plt

# Load color image
image = cv2.imread('dogimg.jpg')

# Reshape image:
# Original shape = height x width x 3
# New shape = number_of_pixels x 3
pixel_values = image.reshape(
    (-1, 3)
)

# Convert pixel values to float32
pixel_values = np.float32(
    pixel_values
)

# Define K-Means stopping criteria
# Stop when:
# 1. Maximum iterations = 100
# OR
# 2. Centroid movement/error becomes less than 0.2
criteria = (
    cv2.TERM_CRITERIA_EPS +
    cv2.TERM_CRITERIA_MAX_ITER,
    100,
    0.2
)

# Number of clusters
K = 4

# Apply K-Means clustering
_, labels, centers = cv2.kmeans(
    pixel_values,              # Pixel data
    K,                         # Number of clusters
    None,                      # Initial labels
    criteria,                  # Stopping criteria
    10,                        # Number of attempts
    cv2.KMEANS_RANDOM_CENTERS  # Random initial centers
)

# Convert cluster centers to uint8
centers = np.uint8(
    centers
)

# Replace each pixel with its cluster center
segmented_image = centers[
    labels.flatten()
]

# Reshape back to original image dimensions
segmented_image = segmented_image.reshape(
    image.shape
)

# Display using OpenCV
cv2.imshow(
    'Segmented Image',
    segmented_image
)

# Wait until a key is pressed
cv2.waitKey(0)

# Close all OpenCV windows
cv2.destroyAllWindows()

# Display using Matplotlib
# Convert BGR to RGB for correct colors
plt.imshow(
    cv2.cvtColor(
        segmented_image,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis('off')
plt.show()

Lab TASKS

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 1: DOCUMENT SEGMENTATION UNDER UNEVEN LIGHTING
# ============================================================

# ------------------------------------------------------------
# 1. LOAD IMAGE IN GRAYSCALE
# ------------------------------------------------------------
# cv2.imread() reads the image.
#
# First parameter:
#     'document.jpg' = image file/path
#
# Second parameter:
#     cv2.IMREAD_GRAYSCALE = load image as a single-channel
#                            grayscale image.
#
# Grayscale pixel values range from:
#     0   = black
#     255 = white
# ------------------------------------------------------------

image = cv2.imread(
    'document.jpg',
    cv2.IMREAD_GRAYSCALE
)

# ------------------------------------------------------------
# 2. APPLY GLOBAL THRESHOLDING
# ------------------------------------------------------------
# We use THREE different threshold values as required.
#
# T1 = 80
# T2 = 128
# T3 = 180
#
# THRESH_BINARY means:
#
#     pixel > threshold  -> 255
#     pixel <= threshold -> 0
#
# cv2.threshold() returns TWO values:
#     1. threshold actually used
#     2. resulting binary image
#
# We do not need the first value, so we use "_".
# ------------------------------------------------------------

_, global_T1 = cv2.threshold(
    image,
    80,
    255,
    cv2.THRESH_BINARY
)

_, global_T2 = cv2.threshold(
    image,
    128,
    255,
    cv2.THRESH_BINARY
)

_, global_T3 = cv2.threshold(
    image,
    180,
    255,
    cv2.THRESH_BINARY
)

# ------------------------------------------------------------
# 3. APPLY ADAPTIVE THRESHOLDING
# ------------------------------------------------------------
# Unlike global thresholding, adaptive thresholding calculates
# a different threshold for different local neighbourhoods.
#
# 11 = blockSize
#      Each local region considered is 11 x 11.
#
# 2 = C
#     Constant subtracted from the calculated local threshold.
#
# GAUSSIAN_C means a weighted Gaussian average is used.
# ------------------------------------------------------------

adaptive = cv2.adaptiveThreshold(
    image,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11,
    2
)

# ------------------------------------------------------------
# 4. DISPLAY ALL RESULTS
# ------------------------------------------------------------
# plt.figure() creates a figure.
#
# subplot(rows, columns, position)
# Here we create 5 images in one row.
# ------------------------------------------------------------

plt.figure(figsize=(18, 5))

plt.subplot(1, 5, 1)
plt.imshow(image, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(1, 5, 2)
plt.imshow(global_T1, cmap='gray')
plt.title('Global T = 80')
plt.axis('off')

plt.subplot(1, 5, 3)
plt.imshow(global_T2, cmap='gray')
plt.title('Global T = 128')
plt.axis('off')

plt.subplot(1, 5, 4)
plt.imshow(global_T3, cmap='gray')
plt.title('Global T = 180')
plt.axis('off')

plt.subplot(1, 5, 5)
plt.imshow(adaptive, cmap='gray')
plt.title('Adaptive')
plt.axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. SAVE FINAL RESULT
# ------------------------------------------------------------
# cv2.imwrite() saves an image to disk.
#
# First parameter:
#     output filename
#
# Second parameter:
#     image to save
# ------------------------------------------------------------

cv2.imwrite(
    'task1_final_adaptive.png',
    adaptive
)

In [ ]:
# TASK 2 — CODE
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 2: ADAPTIVE THRESHOLD PARAMETER STUDY
# ============================================================

# Load the same document in grayscale
image = cv2.imread(
    'document.jpg',
    cv2.IMREAD_GRAYSCALE
)

# ------------------------------------------------------------
# PARAMETERS TO TEST
# ------------------------------------------------------------
# blockSize must be an odd number greater than 1.
#
# We test:
#     7
#     11
#     21
#
# We also test different C values:
#     2
#     5
#     10
# ------------------------------------------------------------

block_sizes = [7, 11, 21]
C_values = [2, 5, 10]

# ------------------------------------------------------------
# CREATE RESULTS
# ------------------------------------------------------------

results = []

# ------------------------------------------------------------
# TEST GAUSSIAN ADAPTIVE THRESHOLDING
# ------------------------------------------------------------

for block_size in block_sizes:

    for C in C_values:

        result = cv2.adaptiveThreshold(
            image,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            block_size,
            C
        )

        results.append(
            (
                result,
                f'Gaussian: B={block_size}, C={C}'
            )
        )

# ------------------------------------------------------------
# DISPLAY 9 DIFFERENT RESULTS
# ------------------------------------------------------------

plt.figure(figsize=(15, 12))

for i in range(len(results)):

    result = results[i][0]
    title = results[i][1]

    plt.subplot(3, 3, i + 1)

    plt.imshow(
        result,
        cmap='gray'
    )

    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# NOW COMPARE MEAN VS GAUSSIAN
# ------------------------------------------------------------

mean_result = cv2.adaptiveThreshold(
    image,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    11,
    5
)

gaussian_result = cv2.adaptiveThreshold(
    image,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11,
    5
)

# ------------------------------------------------------------
# DISPLAY BOTH METHODS
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(mean_result, cmap='gray')
plt.title('Mean Adaptive')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(gaussian_result, cmap='gray')
plt.title('Gaussian Adaptive')
plt.axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# SAVE THE FINAL CHOSEN RESULT
# ------------------------------------------------------------

# After visually inspecting the results,
# select the best combination.
#
# Here we assume blockSize=11 and C=5
# is the best combination.
# Change these values according to your actual result.

final_result = cv2.adaptiveThreshold(
    image,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11,
    5
)

cv2.imwrite(
    'task2_final.png',
    final_result
)

In [ ]:
# TASK 3 — CODE
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 3: OTSU'S THRESHOLDING
# ============================================================

# ------------------------------------------------------------
# 1. LOAD IMAGE
# ------------------------------------------------------------

image = cv2.imread(
    'coins.jpg'
)

# ------------------------------------------------------------
# 2. CONVERT TO GRAYSCALE
# ------------------------------------------------------------

gray = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2GRAY
)

# ------------------------------------------------------------
# 3. GENERATE HISTOGRAM
# ------------------------------------------------------------
# calcHist() calculates the histogram of pixel intensities.
#
# [gray] = input image list
# [0] = channel 0 because grayscale has only one channel
# None = no mask; use complete image
# [256] = 256 possible intensity values
# [0,256] = intensity range
# ------------------------------------------------------------

histogram = cv2.calcHist(
    [gray],
    [0],
    None,
    [256],
    [0, 256]
)

# ------------------------------------------------------------
# 4. APPLY OTSU THRESHOLDING
# ------------------------------------------------------------
# thresh=0 is given because Otsu automatically finds
# the appropriate threshold.
#
# retval contains the threshold selected by Otsu.
# binary contains the resulting binary mask.
# ------------------------------------------------------------

otsu_threshold, binary = cv2.threshold(
    gray,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Print automatically selected threshold
print(
    'Otsu threshold:',
    otsu_threshold
)

# ------------------------------------------------------------
# 5. CHANGE CONTRAST / PREPROCESS IMAGE
# ------------------------------------------------------------
# alpha controls contrast.
# beta controls brightness.
#
# alpha = 1.2 means increase contrast.
# beta = 0 means no brightness shift.
# ------------------------------------------------------------

changed = cv2.convertScaleAbs(
    gray,
    alpha=1.2,
    beta=0
)

# Apply Otsu again
otsu_threshold_2, binary_2 = cv2.threshold(
    changed,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print(
    'Otsu threshold after preprocessing:',
    otsu_threshold_2
)

# ------------------------------------------------------------
# 6. DISPLAY ORIGINAL + HISTOGRAM + OTSU MASK
# ------------------------------------------------------------

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(
    gray,
    cmap='gray'
)
plt.title('Original Grayscale')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.plot(
    histogram
)
plt.title('Grayscale Histogram')
plt.xlabel('Intensity')
plt.ylabel('Frequency')

plt.subplot(1, 3, 3)
plt.imshow(
    binary,
    cmap='gray'
)
plt.title(
    f'Otsu Binary, T={otsu_threshold:.0f}'
)
plt.axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. DISPLAY PREPROCESSED RESULT
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(
    changed,
    cmap='gray'
)
plt.title('Contrast Changed')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(
    binary_2,
    cmap='gray'
)
plt.title(
    f'Otsu After Preprocessing, T={otsu_threshold_2:.0f}'
)
plt.axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. SAVE FINAL OTSU MASK
# ------------------------------------------------------------

cv2.imwrite(
    'task3_otsu.png',
    binary
)

In [ ]:
# TASK 4 — CODE
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 4: COLOR-BASED SEGMENTATION
# ============================================================

# ------------------------------------------------------------
# 1. LOAD IMAGE IN COLOR
# ------------------------------------------------------------
# OpenCV reads color images in BGR format.
# ------------------------------------------------------------

image = cv2.imread(
    'yellow_car.jpg'
)

# ------------------------------------------------------------
# 2. CONVERT BGR TO HSV
# ------------------------------------------------------------
# HSV:
# H = Hue        -> actual color
# S = Saturation -> color purity/intensity
# V = Value      -> brightness
#
# HSV is often easier for color segmentation.
# ------------------------------------------------------------

hsv = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2HSV
)

# ------------------------------------------------------------
# 3. CREATE A TOO-RESTRICTIVE MASK
# ------------------------------------------------------------
# This range is deliberately narrow.
# It may select only part of the desired object.
#
# Format:
# [Hue, Saturation, Value]
# ------------------------------------------------------------

lower_restrictive = np.array(
    [25, 120, 120]
)

upper_restrictive = np.array(
    [35, 255, 255]
)

restrictive_mask = cv2.inRange(
    hsv,
    lower_restrictive,
    upper_restrictive
)

# ------------------------------------------------------------
# 4. CREATE A FINAL / LESS RESTRICTIVE MASK
# ------------------------------------------------------------
# Wider range allows more shades of the desired color.
# ------------------------------------------------------------

lower_final = np.array(
    [20, 70, 70]
)

upper_final = np.array(
    [40, 255, 255]
)

final_mask = cv2.inRange(
    hsv,
    lower_final,
    upper_final
)

# ------------------------------------------------------------
# 5. EXTRACT SELECTED COLOR FROM ORIGINAL IMAGE
# ------------------------------------------------------------
# bitwise_and keeps original pixel values where mask=255.
# Everywhere else becomes black.
# ------------------------------------------------------------

selected_object = cv2.bitwise_and(
    image,
    image,
    mask=final_mask
)

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

plt.figure(figsize=(16, 8))

plt.subplot(2, 3, 1)
plt.imshow(
    cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
)
plt.title('Original')
plt.axis('off')

plt.subplot(2, 3, 2)
plt.imshow(
    restrictive_mask,
    cmap='gray'
)
plt.title('Too Restrictive Mask')
plt.axis('off')

plt.subplot(2, 3, 3)
plt.imshow(
    final_mask,
    cmap='gray'
)
plt.title('Final Mask')
plt.axis('off')

plt.subplot(2, 3, 4)
plt.imshow(
    cv2.cvtColor(
        selected_object,
        cv2.COLOR_BGR2RGB
    )
)
plt.title('Extracted Object')
plt.axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. SAVE FINAL OUTPUT
# ------------------------------------------------------------

cv2.imwrite(
    'task4_final_mask.png',
    final_mask
)

cv2.imwrite(
    'task4_extracted_object.png',
    selected_object
)

In [ ]:
# TASK 5 — CODE
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 5: CANNY EDGE DETECTION
# ============================================================

# ------------------------------------------------------------
# 1. LOAD IMAGE
# ------------------------------------------------------------

image = cv2.imread(
    'mechanical_part.jpg'
)

# Convert to grayscale
gray = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2GRAY
)

# ------------------------------------------------------------
# 2. APPLY CANNY WITH THREE THRESHOLD PAIRS
# ------------------------------------------------------------

# Result 1:
# Lower threshold = 50
# Upper threshold = 150

edges_1 = cv2.Canny(
    gray,
    50,
    150
)

# Result 2:
# Lower threshold = 100
# Upper threshold = 200

edges_2 = cv2.Canny(
    gray,
    100,
    200
)

# Result 3:
# Lower threshold = 100
# Upper threshold = 250

edges_3 = cv2.Canny(
    gray,
    100,
    250
)

# ------------------------------------------------------------
# 3. DISPLAY RESULTS
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

plt.subplot(1, 4, 1)
plt.imshow(
    cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
)
plt.title('Original')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(
    edges_1,
    cmap='gray'
)
plt.title('Canny: 50 / 150')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(
    edges_2,
    cmap='gray'
)
plt.title('Canny: 100 / 200')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(
    edges_3,
    cmap='gray'
)
plt.title('Canny: 100 / 250')
plt.axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. SAVE RESULTS
# ------------------------------------------------------------

cv2.imwrite(
    'task5_edges_1.png',
    edges_1
)

cv2.imwrite(
    'task5_edges_2.png',
    edges_2
)

cv2.imwrite(
    'task5_edges_3.png',
    edges_3
)

In [ ]:
# TASK 6 — CODE
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 6: REGION GROWING
# ============================================================

# ------------------------------------------------------------
# LOAD GRAYSCALE MRI IMAGE
# ------------------------------------------------------------

image = cv2.imread(
    'brain_mri.jpg',
    cv2.IMREAD_GRAYSCALE
)


# ------------------------------------------------------------
# REGION GROWING FUNCTION
# ------------------------------------------------------------

def region_growing(
    image,
    seed,
    threshold
):

    # Create an empty binary mask.
    # Same dimensions as input image.
    #
    # Initially every pixel = 0.
    mask = np.zeros_like(
        image,
        dtype=np.uint8
    )

    # Stack stores pixels that still need to be checked.
    stack = [seed]

    # Get intensity of starting seed pixel.
    seed_intensity = int(
        image[seed]
    )

    # Continue until stack becomes empty.
    while stack:

        # Get one pixel from stack.
        x, y = stack.pop()

        # ----------------------------------------------------
        # CHECK IMAGE BOUNDARIES
        # ----------------------------------------------------

        if (
            x < 0 or
            x >= image.shape[0] or
            y < 0 or
            y >= image.shape[1]
        ):
            continue

        # ----------------------------------------------------
        # CHECK IF PIXEL WAS ALREADY VISITED
        # ----------------------------------------------------

        if mask[x, y] != 0:
            continue

        # ----------------------------------------------------
        # CALCULATE INTENSITY DIFFERENCE
        # ----------------------------------------------------

        difference = abs(
            int(image[x, y]) -
            seed_intensity
        )

        # ----------------------------------------------------
        # ADD PIXEL IF IT IS SIMILAR ENOUGH
        # ----------------------------------------------------

        if difference <= threshold:

            # Mark pixel as part of region
            mask[x, y] = 255

            # Add 4-connected neighbours
            stack.append(
                (x + 1, y)
            )

            stack.append(
                (x - 1, y)
            )

            stack.append(
                (x, y + 1)
            )

            stack.append(
                (x, y - 1)
            )

    return mask


# ------------------------------------------------------------
# TEST DIFFERENT SEED POINTS AND THRESHOLDS
# ------------------------------------------------------------

seed_1 = (100, 100)
seed_2 = (150, 150)

thresholds = [10, 25, 50]

results = []

# ------------------------------------------------------------
# TEST SEED 1
# ------------------------------------------------------------

for threshold in thresholds:

    result = region_growing(
        image,
        seed_1,
        threshold
    )

    results.append(
        (
            result,
            f'Seed 1, T={threshold}'
        )
    )

# ------------------------------------------------------------
# TEST SEED 2
# ------------------------------------------------------------

for threshold in thresholds:

    result = region_growing(
        image,
        seed_2,
        threshold
    )

    results.append(
        (
            result,
            f'Seed 2, T={threshold}'
        )
    )

# ------------------------------------------------------------
# DISPLAY ALL SIX RESULTS
# ------------------------------------------------------------

plt.figure(figsize=(15, 10))

for i in range(len(results)):

    result = results[i][0]
    title = results[i][1]

    plt.subplot(2, 3, i + 1)

    plt.imshow(
        result,
        cmap='gray'
    )

    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# TASK 7 — COMPLETE CODE
import cv2
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# TASK 7: MARKER-BASED WATERSHED SEGMENTATION
# ============================================================

# ------------------------------------------------------------
# 1. LOAD IMAGE
# ------------------------------------------------------------

image = cv2.imread(
    'coins.jpg'
)

# Keep a copy for final visualization
original = image.copy()

# ------------------------------------------------------------
# 2. CONVERT TO GRAYSCALE
# ------------------------------------------------------------

gray = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2GRAY
)

# ------------------------------------------------------------
# 3. THRESHOLDING
# ------------------------------------------------------------
# Otsu automatically selects threshold.
# Binary inverse is used depending on the coin/background
# polarity in the image.
# ------------------------------------------------------------

_, thresh = cv2.threshold(
    gray,
    0,
    255,
    cv2.THRESH_BINARY_INV +
    cv2.THRESH_OTSU
)

# ------------------------------------------------------------
# 4. NOISE REMOVAL
# ------------------------------------------------------------
# Create morphological kernel.
# 3x3 is used here.
# ------------------------------------------------------------

kernel = np.ones(
    (3, 3),
    np.uint8
)

# Opening = erosion followed by dilation.
# It removes small noise.
opening = cv2.morphologyEx(
    thresh,
    cv2.MORPH_OPEN,
    kernel,
    iterations=2
)

# ------------------------------------------------------------
# 5. SURE BACKGROUND
# ------------------------------------------------------------
# Dilation expands the foreground/background region.
# Here it is used to create a region that is definitely
# background.
# ------------------------------------------------------------

sure_bg = cv2.dilate(
    opening,
    kernel,
    iterations=3
)

# ------------------------------------------------------------
# 6. DISTANCE TRANSFORM
# ------------------------------------------------------------
# Distance transform gives the distance of foreground pixels
# from the nearest background pixel.
#
# Center of an object -> large distance
# Boundary           -> small distance
# ------------------------------------------------------------

dist_transform = cv2.distanceTransform(
    opening,
    cv2.DIST_L2,
    5
)

# ------------------------------------------------------------
# 7. SURE FOREGROUND
# ------------------------------------------------------------
# Only pixels sufficiently far from the background
# are considered definite object/foreground.
#
# 0.2 * maximum distance is the selected threshold.
# ------------------------------------------------------------

_, sure_fg = cv2.threshold(
    dist_transform,
    0.2 * dist_transform.max(),
    255,
    0
)

# Convert to uint8
sure_fg = np.uint8(
    sure_fg
)

# ------------------------------------------------------------
# 8. UNKNOWN REGION
# ------------------------------------------------------------
# Background - foreground
# gives the area where we are not sure whether it is
# foreground or background.
# ------------------------------------------------------------

unknown = cv2.subtract(
    sure_bg,
    sure_fg
)

# ------------------------------------------------------------
# 9. MARKER LABELING
# ------------------------------------------------------------
# Each connected foreground object receives a label.
# ------------------------------------------------------------

num_labels, markers = cv2.connectedComponents(
    sure_fg
)

print(
    'Number of connected foreground components:',
    num_labels - 1
)

# ------------------------------------------------------------
# 10. SHIFT MARKERS
# ------------------------------------------------------------
# Background label 0 becomes 1.
# This keeps 0 available for unknown region.
# ------------------------------------------------------------

markers = markers + 1

# Unknown pixels are assigned marker 0.
markers[unknown == 255] = 0

# ------------------------------------------------------------
# 11. APPLY WATERSHED
# ------------------------------------------------------------

cv2.watershed(
    image,
    markers
)

# ------------------------------------------------------------
# 12. VISUALIZE WATERSHED BOUNDARIES
# ------------------------------------------------------------
# Watershed marks boundary pixels as -1.
#
# OpenCV uses BGR:
# [0, 0, 255] = RED
# ------------------------------------------------------------

image[markers == -1] = [
    0,
    0,
    255
]

# ------------------------------------------------------------
# 13. DISPLAY ALL REQUIRED STAGES
# ------------------------------------------------------------

plt.figure(figsize=(18, 12))

plt.subplot(2, 4, 1)
plt.imshow(
    cv2.cvtColor(
        original,
        cv2.COLOR_BGR2RGB
    )
)
plt.title('Original')
plt.axis('off')

plt.subplot(2, 4, 2)
plt.imshow(
    thresh,
    cmap='gray'
)
plt.title('Threshold')
plt.axis('off')

plt.subplot(2, 4, 3)
plt.imshow(
    sure_bg,
    cmap='gray'
)
plt.title('Sure Background')
plt.axis('off')

plt.subplot(2, 4, 4)
plt.imshow(
    dist_transform,
    cmap='gray'
)
plt.title('Distance Transform')
plt.axis('off')

plt.subplot(2, 4, 5)
plt.imshow(
    sure_fg,
    cmap='gray'
)
plt.title('Sure Foreground')
plt.axis('off')

plt.subplot(2, 4, 6)
plt.imshow(
    unknown,
    cmap='gray'
)
plt.title('Unknown Region')
plt.axis('off')

plt.subplot(2, 4, 7)
plt.imshow(
    cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
)
plt.title('Final Watershed')
plt.axis('off')

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 14. SAVE FINAL OUTPUT
# ------------------------------------------------------------

cv2.imwrite(
    'task7_watershed.png',
    image
)